# 01 · Exploratory analysis

Questions to answer before modeling: how rare is fraud, where is it concentrated, what does the identity
LEFT join change, how do amounts and timing differ, and what does missingness tell us?

All heavy lifting is SQL against the DuckDB file built by `make features`; this notebook only displays
results. Set `FRAUD_PROFILE=sample` to run it on synthetic data.

In [1]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

profile: full


In [2]:
conn = db.connect(paths.db, read_only=True)
conn.execute("SET enable_progress_bar = false")
q = lambda sql, **kw: conn.execute(db.read_sql(sql, **kw) if sql.endswith(".sql") else sql).df()
support = 500 if PROFILE == "full" else 30

## 1. Class balance and where fraud concentrates

In [3]:
overall = q("SELECT count(*) AS n, avg(isFraud) AS fraud_rate, "
            "sum(TransactionAmt) FILTER (WHERE isFraud = 1) / sum(TransactionAmt) AS fraud_usd_share FROM features")
overall

,n,fraud_rate,fraud_usd_share
0,590540,0.03499,0.038674


In [4]:
cohorts = q("analysis/01_fraud_rate_by_cohort.sql", min_support=support)
cohorts

,dimension,value,transactions,fraud_rate,lift,fraud_usd
0,DeviceType,mobile,55645,0.1017,2.91,546427.0
1,DeviceType,desktop,85165,0.0652,1.86,449879.0
2,DeviceType,(no identity),449730,0.0210,0.60,2087538.0
3,ProductCD,C,68519,0.1169,3.34,391421.0
4,ProductCD,S,11628,0.0590,1.69,43416.0
5,ProductCD,H,33024,0.0477,1.36,246632.0
6,ProductCD,R,37699,0.0378,1.08,348050.0
7,ProductCD,W,439670,0.0204,0.58,2054325.0
8,card4,discover,6651,0.0773,2.21,182000.0
9,card4,visa,384767,0.0348,0.99,1997706.0


In [5]:
base_rate = float(overall.fraud_rate[0])
bars = alt.Chart(cohorts).mark_bar(color=charts.SERIES[0], cornerRadiusEnd=4, height=14).encode(
    y=alt.Y("value:N", title=None, sort="-x"), x=alt.X("fraud_rate:Q", axis=alt.Axis(format="%"), title="Fraud rate"),
    tooltip=["dimension", "value", "transactions", alt.Tooltip("fraud_rate:Q", format=".2%"), "lift"])
rule = alt.Chart(pd.DataFrame({"x": [base_rate]})).mark_rule(strokeDash=[4, 4], color=charts.MUTED).encode(x="x:Q")
alt.layer(bars, rule).properties(width=320, height=alt.Step(18)).facet(
    row=alt.Row("dimension:N", title=None), data=cohorts).resolve_scale(y="independent").properties(
    title=alt.Title("Fraud rate by cohort", subtitle="dashed = overall fraud rate"))

alt.FacetChart(...)

## 2. What an INNER join to identity would have done

In [6]:
q("analysis/02_identity_missingness.sql")

,population,transactions,share_of_rows,fraud_rate,fraud_usd,share_of_fraud_usd,median_amount
0,kept by INNER join (has identity),144233,0.2442,0.0785,1005155.0,0.3259,50.00
1,dropped by INNER join (no identity),446307,0.7558,0.0209,2078689.0,0.6741,77.95


**Finding (DRAFT, rewrite in your voice after running):** _N% of transactions have no identity row. Those
rows carry X% of fraud dollars at a fraud rate of Y% vs Z% with identity. An INNER join would have thrown
them away and changed the base rate the model learns._

## 3. Amounts: fraud vs legitimate

In [7]:
amt = q("SELECT isFraud, TransactionAmt, amt_cents FROM features")
amt["class"] = np.where(amt.isFraud == 1, "fraud", "legitimate")
# aggregate before charting so the notebook embeds ~120 rows, not the raw data
edges = np.linspace(np.log10(0.25), np.log10(35000), 61)
amt["bin"] = pd.cut(np.log10(amt.TransactionAmt), edges, labels=edges[:-1].round(3))
hist = amt.groupby(["class", "bin"], observed=True).size().rename("n").reset_index()
hist["share"] = hist.n / hist.groupby("class").n.transform("sum")
hist["bin"] = hist.bin.astype(float)
alt.Chart(hist).mark_line(interpolate="step-after").encode(
    x=alt.X("bin:Q", title="log10(amount, USD)"), y=alt.Y("share:Q", title="Share of class", axis=alt.Axis(format="%")),
    color=alt.Color("class:N", scale=alt.Scale(domain=["legitimate", "fraud"], range=charts.SERIES[:2])),
    tooltip=["class", alt.Tooltip("bin:Q", format=".2f"), alt.Tooltip("share:Q", format=".2%")]
).properties(width=560, height=260, title="Amount distribution by class (each class sums to 100%)")

alt.Chart(...)

In [8]:
amt["cents_band"] = pd.cut(amt.amt_cents, [-0.001, 0.0005, 0.5, 0.9495, 1], labels=["$.00", "$.01-.50", "$.51-.95", "$.95-.99"])
amt.groupby("cents_band", observed=True).agg(n=("isFraud", "size"), fraud_rate=("isFraud", "mean"))

,n,fraud_rate
cents_band,,
$.00,305013,0.035677
$.01-.50,64514,0.067877
$.51-.95,36116,0.102060
$.95-.99,184897,0.009281


## 4. Timing (relative hour; the reference date is unknown)

In [9]:
hourly = q("SELECT hour, count(*) AS n, avg(isFraud) AS fraud_rate FROM features GROUP BY hour ORDER BY hour")
alt.Chart(hourly).mark_line(point=True, color=charts.SERIES[0]).encode(
    x=alt.X("hour:O", title="Hour of day (relative to unknown reference)"),
    y=alt.Y("fraud_rate:Q", axis=alt.Axis(format="%"), title="Fraud rate"),
    tooltip=["hour", "n", alt.Tooltip("fraud_rate:Q", format=".2%")]).properties(width=560, height=240)

alt.Chart(...)

_Hour 0 is not midnight: the time axis has no calendar anchor, so only relative patterns count._

## 5. Missingness as a signal

In [10]:
cols = [c for c in conn.execute("DESCRIBE base").df().column_name if c not in ("TransactionID", "isFraud")]
exprs = ", ".join(f"avg(CAST({c} IS NULL AS DOUBLE)) AS \"{c}\"" for c in cols)
miss = conn.execute(f"SELECT {exprs} FROM base").df().T.rename(columns={0: "missing_share"})
print("columns >90% missing:", int((miss.missing_share > 0.9).sum()), "of", len(miss))
probe = [c for c in miss.index if 0.05 < miss.missing_share[c] < 0.95]
rows = []
for c in probe:
    r = conn.execute(f"SELECT avg(isFraud) FILTER (WHERE {c} IS NULL), avg(isFraud) FILTER (WHERE {c} IS NOT NULL) FROM base").fetchone()
    rows.append({"column": c, "missing_share": miss.missing_share[c], "fraud_if_missing": r[0], "fraud_if_present": r[1]})
mt = pd.DataFrame(rows)
mt["ratio"] = mt.fraud_if_present / mt.fraud_if_missing
mt.sort_values("ratio", ascending=False).head(15)

columns >90% missing: 12 of 436


,column,missing_share,fraud_if_missing,fraud_if_present,ratio
11,D7,0.934099,0.026962,0.148778,5.518017
16,D12,0.890410,0.024847,0.117403,4.725101
18,D14,0.894695,0.025456,0.115989,4.556362
17,D13,0.895093,0.026156,0.110360,4.219209
10,D6,0.876068,0.025022,0.105456,4.214594
289,id_10,0.873123,0.024895,0.104463,4.196215
13,D9,0.873123,0.024895,0.104463,4.196215
288,id_09,0.873123,0.024895,0.104463,4.196215
12,D8,0.873123,0.024895,0.104463,4.196215
285,id_04,0.887689,0.025850,0.107231,4.148203


## 6. Drift across the time split

In [11]:
pd.DataFrame(load_json(paths.reports / "train_results.json")["splits"])

,split,rows,fraud_rate,fraud_usd,first_day,last_day
0,train,354324,0.033833,1677790.677,1,101
1,valid,88581,0.040359,615010.657,101,130
2,calib,59054,0.034155,321435.005,130,152
3,test,88581,0.034804,469608.521,152,182


**Findings (DRAFT, rewrite after running):**
- _Base rate and how it drifts across splits_
- _Highest-risk cohorts (product, device, identity) and their lift_
- _Amount / cents pattern_
- _Which missingness flags carry signal_